# Collision experiment — combined dataset preparation
Run `raw_data_preparation_collision.ipynb` first whenever a recording does not yet contain `XY_full.h5`. This notebook selects the Collision sessions, parses their conditions, aligns the trajectories and creates the combined pickle used by the downstream analysis.

### 0. Initialisation

#### 0.1 Locate the workflow and define data paths
The setup locates the repository automatically, uses `analysis_methods_collision.json`, points to the mounted LocustVR data, and defines where `locustvr_data.pkl` is saved. Set `save_pickle = False` only when the combined dataset should not be overwritten.

In [ ]:
from pathlib import Path

current_working_directory = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in [current_working_directory, *current_working_directory.parents]
     if (candidate / 'locustvr_extractor.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate the UnityDataAnalysis repository root.')
workflow_dir = repo_root / 'analyses' / 'collision'
root_dir_data = "/Volumes/AG_Couzin-Fuchs/DATA/experiment_trackball_Optomotor/locustVR"
save_pickle = True
save_path = "/Users/jonny/Desktop/Uni/Master/collevtive behavior/VR_analysis/sequence_choice"

#### 0.2 Import packages

In [ ]:
import os,json
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
plt.style.use('default')
import importlib
import sys
from pathlib import Path

utilities_dir = repo_root.parent / 'utilities'
for import_dir in (repo_root, workflow_dir, utilities_dir):
    import_path = str(import_dir)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

from LocustVR_data_analysis import utils_local

importlib.reload(utils_local)
from LocustVR_data_analysis.utils_local import align_and_flip_heading, reindex_t_by_state_transition, convert_trial_label, align_trajectories, \
    compute_directness_and_direction, exclude_flagged_animals, flip_symmetric_states

from useful_tools import select_animals_gpt,find_file,column_name_list,get_fill_between_range,read_seq_config
from data_cleaning import findLongestConseqSubseq,interp_fill

json_file = workflow_dir / 'analysis_methods_collision.json'

with open(json_file, "r") as f:
    analysis_methods = json.loads(f.read())

variable_name='location'
exp_name=analysis_methods.get("experiment_name")

### 1. Data preparation

#### 1.1 Select Collision sessions
The Google Sheet supplies valid animal IDs. Animals marked `T` in the general exclusion column are removed by default. The 24- and cumulative 48-hour molting filters can be enabled independently in the analysis settings. The additional GN-ID filter retains Collision animals from `GN25074` through `GN25118`. Inspect `dir_lis_2026` before loading the files.

In [ ]:
# Define the path to your Excel file
dir_list = []
file_type=".h5"
using_google_sheet=True
sheet_name = 'LocustVR'
EXCLUSION_COLUMN = 'Excluding this animal from analysis (Usually when animals die or molt, T/F)'
MOLTING_24H_COLUMN = 'Molting after 24 hour?'
MOLTING_48H_COLUMN = 'Molting after 48 hour?'
load_individual_data = analysis_methods.get('load_individual_data', False)
exclude_general = analysis_methods.get('exclude_animals_marked_for_exclusion', True)
exclude_molting_24h = analysis_methods.get('exclude_molting_after_24h', False)
exclude_molting_48h = analysis_methods.get('exclude_molting_after_48h', False)
if load_individual_data or exclude_general or exclude_molting_24h or exclude_molting_48h:
    if using_google_sheet==True:
        # database_id = "1UL4eEUrQMapx9xz11-IyOSlPBcep3I9vBJ2uGgVudb8"
        #         #https://docs.google.com/spreadsheets/d/1UL4eEUrQMapx9xz11-IyOSlPBcep3I9vBJ2uGgVudb8/edit?usp=sharing
        # url = f"https://docs.google.com/spreadsheets/d/{database_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
        database_id = "1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8"
                #https://docs.google.com/spreadsheets/d/1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8/edit?usp=sharing
        url = f"https://docs.google.com/spreadsheets/d/{database_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
        #df = pd.read_excel(url, engine='openpyxl')## use this function if the file is not google sheet but uploaded excel file
        df = pd.read_csv(url)
    else:
        excel_file_path = "Z:/DATA/experiment_trackball_Optomotor/Locusts Management.xlsx"
        print(f"using a database {excel_file_path} from the server but this file might be outdated")
        # Create a 'with' statement to open and read the Excel file
        with pd.ExcelFile(excel_file_path) as xls:
            # Read the Excel sheet into a DataFrame with the sheet name (folder name)
            df = pd.read_excel(xls, sheet_name)
        ##list up the conditions and answers as strings for input argument to select animal. One condition must pair with one answer
    animal_of_interest=df
    animal_of_interest = exclude_flagged_animals(
        animal_of_interest,
        EXCLUSION_COLUMN,
        enabled=exclude_general,
    )
    animal_of_interest = exclude_flagged_animals(
        animal_of_interest,
        MOLTING_24H_COLUMN,
        enabled=exclude_molting_24h,
    )
    animal_of_interest = exclude_flagged_animals(
        animal_of_interest,
        MOLTING_48H_COLUMN,
        enabled=exclude_molting_48h,
    )

    ID_array = animal_of_interest["ID"].values
    print(f"Valid animals IDs: \n {ID_array}")
    
    dir_list = [
    root.replace("\\", "/")
    for root, _, files in os.walk(root_dir_data)
    if any(ID in root for ID in ID_array)
    and any(file.endswith(file_type) for file in files)]
else:
    for root, dirs, files in os.walk(root_dir_data):
        for folder in dirs:
            folder_path=os.path.join(root,folder)
            if any(name.endswith(file_type) for name in os.listdir(folder_path)):
                dir_list.append(folder_path.replace("\\", "/"))
dir_list.sort()
print("Directories:")
for i in dir_list:
    print(i)


In [ ]:
import re

filtered_dir_list = [
    path for path in dir_list
    if any(
        25074 <= int(part[2:]) <= 25118
        for part in path.replace("\\", "/").split("/")
        if re.fullmatch(r"GN\d{5}", part)
    )
]

for path in filtered_dir_list:
    print(path)

dir_lis_2026 = filtered_dir_list

In [ ]:
len(dir_lis_2026)

#### 1.2 Build the combined Collision dataset
Each `XY_full.h5` is reindexed by trial, aligned to trial onset and parsed with the Collision label format `Texture…_CS…`. The resulting animal tables are concatenated and optionally saved as `locustvr_data.pkl`.


In [ ]:
file_name = "/XY_full.h5"
pd_list = []
for index, this_dir in enumerate(dir_lis_2026):
    print(index, this_dir)
    this_pd = pd.read_hdf(this_dir + file_name)
    this_pd['animal_id'] = index
    this_pd = reindex_t_by_state_transition(this_pd, transition_from=0, transition_to=(1, 2))
    this_pd = align_and_flip_heading(this_pd)
    this_pd = convert_trial_label(this_pd, exp_type='collision')
    this_pd = align_trajectories(this_pd)
    this_pd = flip_symmetric_states(this_pd)
    this_pd = compute_directness_and_direction(this_pd)
    pd_list.append(this_pd)

df=pd.concat(pd_list,ignore_index=True)

if save_pickle:
    os.makedirs(save_path, exist_ok=True)
    df.to_pickle(os.path.join(save_path, 'locustvr_data.pkl'))

### 2. Inspect an existing combined dataset
These cells reload the saved pickle and summarize the number of trials per animal.

In [ ]:
df = pd.read_pickle(os.path.join(save_path, 'locustvr_data.pkl'))


In [ ]:
df.head(20)


In [ ]:
grouped = df.groupby("animal_id")["trial_id"].agg(["max", "nunique"])
print(grouped)